# Collocation Points and Runge-Kutta Coefficients
**Prepared by:** Prof. Alexander Dowling

This notebook is the computational companion to the collocation lecture. Rather than copying the
collocation points and Butcher coefficients out of a table, we **derive them**: the collocation
points come out of an orthogonality condition, the basis polynomials come out of a symbolic
integral, and the coefficient table comes out of evaluating that integral. Every number is then
checked against the published values.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !wget "https://raw.githubusercontent.com/ndcbe/optimization/main/notebooks/helper.py"
    import helper

    helper.easy_install()
else:
    sys.path.insert(0, "../")
    import helper
helper.set_plotting_style()

import numpy as np
import sympy as sym
import matplotlib.pyplot as plt

# The normalized coordinate on one finite element, tau in [0, 1]
tau = sym.symbols("tau", real=True)

## Notebook context
This notebook follows Chapter 10 of [Biegler (2010)](https://epubs.siam.org/doi/book/10.1137/1.9780898719383).
It assumes the quadrature background in [](./PyomoDAE_theory.ipynb) and the Runge-Kutta notation in
[](./DAE_numeric_integration.ipynb).

Three questions, answered in order:

1. Where do the **basis polynomials** $\bar\ell_j(\tau)$ and $\Omega_j(\tau)$ come from?
2. Where do the **collocation points** $\tau_j$ come from, for each of the three families?
3. How do those two produce the **Runge-Kutta (Butcher) coefficients** $a_{j,k}$ and $b_k$?

## The two Lagrange bases

On finite element $i$ we write $t = t_{i-1} + h_i \tau$ with $\tau \in [0,1]$. There are **two**
Lagrange bases in play, and they are not the same object.

The **state** is interpolated through $K+1$ points, including $\tau_0 = 0$:

\begin{equation}
z^K(t) = \sum_{j=0}^K \ell_j(\tau) z_{ij},
\qquad
\ell_j(\tau) = \prod_{\substack{k=0 \\ k \neq j}}^K \frac{\tau - \tau_k}{\tau_j - \tau_k}.
\end{equation}

The **derivative** is interpolated through only the $K$ collocation points, and $\tau_0 = 0$ is
*not* one of them:

\begin{equation}
\bar\ell_j(\tau) = \prod_{\substack{k=1 \\ k \neq j}}^K \frac{\tau - \tau_k}{\tau_j - \tau_k},
\qquad
\Omega_j(\tau) = \int_0^{\tau} \bar\ell_j(\tau')\, d\tau',
\qquad j = 1, \ldots, K,
\end{equation}

which gives the Runge-Kutta representation of the state,

\begin{equation}
z^K(t) = z_{i-1} + h_i \sum_{j=1}^K \Omega_j(\tau)\, \dot{z}_{ij}.
\end{equation}

```{important}
**$\bar\ell_j$ under the integral, not $\ell_j$.** The printed text of Biegler (2010) defines
$\Omega_j(\tau) = \int_0^{\tau} \ell_j(\tau')\,d\tau'$ with the $(K+1)$-point basis. That is a
typo, corrected in [`errata2.pdf`](https://epubs.siam.org/doi/book/10.1137/1.9780898719383)
(Chapter 10, correction 1). The correct integrand is the $K$-point basis $\bar\ell_j$.

The tell is the degree. $\bar\ell_j$ has degree $K-1$, so $\Omega_j$ has degree $K$ --- which is
what the book says a few lines earlier. With $\ell_j$ (degree $K$) under the integral $\Omega_j$
would come out degree $K+1$, and the collocation equations would no longer match. We check both
degrees numerically below, so the erratum is a computation here rather than a claim.
```

### Building $\bar\ell_j$ and $\Omega_j$ symbolically

Both definitions are products and one integral, so `sympy` can carry them exactly. We keep the
collocation points as exact numbers (rationals and surds) all the way through, and convert to
floating point only at the very end.

In [ ]:
def lagrange_basis(points, j):
    """Lagrange basis polynomial through `points`, equal to 1 at points[j] and 0 elsewhere.

    Arguments:
        points: list of interpolation points (sympy expressions)
        j: index of the point where this basis polynomial equals 1

    Returns:
        sympy expression in tau
    """
    expr = sym.Integer(1)
    for k, tau_k in enumerate(points):
        if k != j:
            expr *= (tau - tau_k) / (points[j] - tau_k)
    return sym.expand(expr)


def omega_basis(points):
    """Runge-Kutta basis polynomials Omega_j(tau) = int_0^tau ellbar_j(tau') dtau'.

    Arguments:
        points: the K collocation points tau_1, ..., tau_K (tau_0 = 0 is NOT included)

    Returns:
        list of K sympy expressions in tau
    """
    return [
        sym.expand(sym.integrate(lagrange_basis(points, j), (tau, 0, tau)))
        for j in range(len(points))
    ]

## Where the collocation points come from

The exact solution of $\dot z = f(z,t)$ on one element is

$$z(t_i) = z(t_{i-1}) + \int_{t_{i-1}}^{t_i} f(z(t), t)\, dt,$$

and collocation replaces that integral with a $K$-point quadrature rule. A rule with $K$ free nodes
and $K$ free weights has $2K$ degrees of freedom, so the best possible rule is exact for
polynomials of degree $2K-1$. Theorem 10.1 of Biegler says the nodes that achieve it are the
**roots of a degree-$K$ polynomial orthogonal on $[0,1]$**.

Constraining an endpoint to be a node spends a degree of freedom and buys a property. That is the
whole difference between the three families:

| Family | Fixed nodes | Free nodes | Weight $w(\tau)$ | Jacobi $(\alpha, \beta)$ | Truncation error |
| --- | --- | --- | --- | --- | --- |
| Gauss-Legendre | none | $K$ | $1$ | $(0, 0)$ | $O(h^{2K})$ |
| Gauss-Radau | $\tau_K = 1$ | $K-1$ | $(1-\tau)$ | $(1, 0)$ | $O(h^{2K-1})$ |
| Gauss-Lobatto | $\tau_1 = 0$, $\tau_K = 1$ | $K-2$ | $\tau(1-\tau)$ | $(1, 1)$ | $O(h^{2K-2})$ |

The free nodes are the roots of the monic polynomial $P_n(\tau)$ of degree $n$ (the number of free
nodes) satisfying the Gauss-Jacobi orthogonality condition

$$\int_0^1 w(\tau)\, \tau^m\, P_n(\tau)\, d\tau = 0, \qquad m = 0, \ldots, n-1.$$

We build $P_n$ by Gram-Schmidt on the monomials $1, \tau, \tau^2, \ldots$ using that weighted inner
product. This is not the numerically preferred route for large $K$ --- the Golub-Welsch
eigenvalue algorithm is --- but it is exact, it is short, and it makes the orthogonality condition
visible instead of hiding it in a library call.

In [ ]:
def monic_orthogonal(n, weight):
    """Monic degree-n polynomial orthogonal to all lower degrees on [0, 1].

    Gram-Schmidt on the monomials under the inner product
    <p, q> = int_0^1 weight(tau) p(tau) q(tau) dtau.

    Arguments:
        n: degree of the polynomial to return
        weight: sympy expression in tau, the Gauss-Jacobi weight function

    Returns:
        sympy expression in tau
    """
    basis = []
    for m in range(n + 1):
        p = tau**m
        for q in basis:
            numer = sym.integrate(weight * p * q, (tau, 0, 1))
            denom = sym.integrate(weight * q * q, (tau, 0, 1))
            p = p - (numer / denom) * q
        basis.append(sym.expand(sym.simplify(p)))
    return basis[n]


# Fixed nodes and Gauss-Jacobi weight for each family
FAMILIES = {
    "Gauss-Legendre": {"fixed": [], "weight": sym.Integer(1)},
    "Gauss-Radau": {"fixed": [sym.Integer(1)], "weight": 1 - tau},
    "Gauss-Lobatto": {"fixed": [sym.Integer(0), sym.Integer(1)], "weight": tau * (1 - tau)},
}


def collocation_points(family, K):
    """Collocation points tau_1 < ... < tau_K on [0, 1], derived from orthogonality.

    Arguments:
        family: one of the keys of FAMILIES
        K: number of collocation points

    Returns:
        sorted list of K exact sympy expressions
    """
    fixed = FAMILIES[family]["fixed"]
    weight = FAMILIES[family]["weight"]
    n_free = K - len(fixed)
    assert n_free >= 0, f"{family} needs K >= {len(fixed)}"

    if n_free > 0:
        P = monic_orthogonal(n_free, weight)
        free = sym.Poly(P, tau).all_roots()
    else:
        free = []

    return sorted([sym.simplify(r) for r in free] + fixed, key=float)

### Reproducing Biegler Table 10.1

The table below is generated, not transcribed. Compare it to Table 10.1 on p. 292 of Biegler (2010).

In [ ]:
COLLOCATION_POINTS = {}

for family in FAMILIES:
    K_min = len(FAMILIES[family]["fixed"])
    print(f"\n{family}")
    for K in range(max(K_min, 1), 6):
        points = collocation_points(family, K)
        COLLOCATION_POINTS[(family, K)] = points
        values = "  ".join(f"{float(p):.6f}" for p in points)
        print(f"  K = {K}:  {values}")

Two of these values are quoted directly in the lecture, so we assert them. If a solver or `sympy`
version ever changes an answer here, the notebook fails loudly instead of quietly publishing a
wrong table.

In [ ]:
# Gauss-Radau, K = 3 -- the points you type into a model (Biegler Table 10.1, p. 292)
radau3 = [float(p) for p in COLLOCATION_POINTS[("Gauss-Radau", 3)]]
assert np.allclose(radau3, [0.155051, 0.644949, 1.0], atol=1e-6), radau3

# Gauss-Legendre, K = 3 -- note the largest root is NOT 1, which is why the
# Legendre continuity equation needs Omega_k(1) rather than a_{Nc,k}
legendre3 = [float(p) for p in COLLOCATION_POINTS[("Gauss-Legendre", 3)]]
assert np.isclose(max(legendre3), 0.887298, atol=1e-6), legendre3

# Exact (radical) forms, for the record
print("Gauss-Radau,     K = 3:", COLLOCATION_POINTS[("Gauss-Radau", 3)])
print("Gauss-Legendre,  K = 3:", COLLOCATION_POINTS[("Gauss-Legendre", 3)])
print("Gauss-Lobatto,   K = 3:", COLLOCATION_POINTS[("Gauss-Lobatto", 3)])

```{note}
The exact forms are worth reading. Gauss-Radau at $K=3$ gives $\tau = \frac{4 \mp \sqrt{6}}{10}$
and $1$; Gauss-Legendre at $K=3$ gives $\frac{1}{2} \pm \frac{1}{2}\sqrt{3/5}$ and $\frac12$. The
six-decimal values in the textbook are these numbers rounded.
```

## Building the coefficient table

With the points in hand, the Runge-Kutta (Butcher) coefficients are just evaluations of $\Omega_k$:

$$c_j = \tau_j, \qquad a_{j,k} = \Omega_k(\tau_j), \qquad b_k = \Omega_k(1).$$

That is the entire content of the tableau. Nothing else is fitted or tuned.

In [ ]:
def butcher_tableau(family, K):
    """Derive the Runge-Kutta (Butcher) coefficients for a collocation family.

    Arguments:
        family: one of the keys of FAMILIES
        K: number of collocation points (stages)

    Returns:
        c: list of K exact collocation points
        A: K-by-K nested list, A[j][k] = Omega_k(tau_j)
        b: list of K exact quadrature weights, b[k] = Omega_k(1)
        Omega: the K basis polynomials, for plotting and degree checks
    """
    c = COLLOCATION_POINTS.get((family, K)) or collocation_points(family, K)
    Omega = omega_basis(c)

    A = [[sym.simplify(Omega[k].subs(tau, c[j])) for k in range(K)] for j in range(K)]
    b = [sym.simplify(Omega[k].subs(tau, 1)) for k in range(K)]

    return c, A, b, Omega


def print_tableau(family, K):
    """Print the Butcher tableau in the usual c | A / b layout."""
    c, A, b, _ = butcher_tableau(family, K)
    print(f"{family}, K = {K}")
    for j in range(K):
        row = "  ".join(f"{float(a):>12.8f}" for a in A[j])
        print(f"  {float(c[j]):.8f} |{row}")
    print("  " + "-" * (12 + 14 * K))
    print("             |" + "  ".join(f"{float(w):>12.8f}" for w in b))


print_tableau("Gauss-Radau", 3)

This is the 3-stage Radau IIA tableau. In exact arithmetic the entries are

In [ ]:
c, A, b, Omega_radau3 = butcher_tableau("Gauss-Radau", 3)

for j in range(3):
    for k in range(3):
        print(f"a_({j + 1},{k + 1}) = {sym.nsimplify(sym.radsimp(A[j][k]))}")
for k in range(3):
    print(f"b_{k + 1}     = {sym.nsimplify(sym.radsimp(b[k]))}")

### Verification against the published tableaus

The 3-stage Radau IIA, Gauss (Legendre) and Lobatto IIIA tableaus are standard. We check every
entry symbolically, so a match is exact rather than to six digits.

In [ ]:
sqrt6 = sym.sqrt(6)
sqrt15 = sym.sqrt(15)

# Radau IIA, 3 stages (Hairer & Wanner; equivalently Biegler Example 10.2)
A_radau_published = [
    [sym.Rational(11, 45) - 7 * sqrt6 / 360, sym.Rational(37, 225) - 169 * sqrt6 / 1800, -sym.Rational(2, 225) + sqrt6 / 75],
    [sym.Rational(37, 225) + 169 * sqrt6 / 1800, sym.Rational(11, 45) + 7 * sqrt6 / 360, -sym.Rational(2, 225) - sqrt6 / 75],
    [sym.Rational(4, 9) - sqrt6 / 36, sym.Rational(4, 9) + sqrt6 / 36, sym.Rational(1, 9)],
]
b_radau_published = A_radau_published[2]

# Gauss (Legendre), 3 stages
A_gauss_published = [
    [sym.Rational(5, 36), sym.Rational(2, 9) - sqrt15 / 15, sym.Rational(5, 36) - sqrt15 / 30],
    [sym.Rational(5, 36) + sqrt15 / 24, sym.Rational(2, 9), sym.Rational(5, 36) - sqrt15 / 24],
    [sym.Rational(5, 36) + sqrt15 / 30, sym.Rational(2, 9) + sqrt15 / 15, sym.Rational(5, 36)],
]
b_gauss_published = [sym.Rational(5, 18), sym.Rational(4, 9), sym.Rational(5, 18)]

# Lobatto IIIA, 3 stages
A_lobatto_published = [
    [sym.Integer(0), sym.Integer(0), sym.Integer(0)],
    [sym.Rational(5, 24), sym.Rational(1, 3), -sym.Rational(1, 24)],
    [sym.Rational(1, 6), sym.Rational(2, 3), sym.Rational(1, 6)],
]
b_lobatto_published = [sym.Rational(1, 6), sym.Rational(2, 3), sym.Rational(1, 6)]

PUBLISHED = {
    "Gauss-Radau": (A_radau_published, b_radau_published),
    "Gauss-Legendre": (A_gauss_published, b_gauss_published),
    "Gauss-Lobatto": (A_lobatto_published, b_lobatto_published),
}

for family, (A_ref, b_ref) in PUBLISHED.items():
    _, A_computed, b_computed, _ = butcher_tableau(family, 3)
    for j in range(3):
        for k in range(3):
            assert (
                sym.simplify(A_computed[j][k] - A_ref[j][k]) == 0
            ), f"{family}: a_({j + 1},{k + 1}) disagrees"
    for k in range(3):
        assert sym.simplify(b_computed[k] - b_ref[k]) == 0, f"{family}: b_{k + 1} disagrees"
    print(f"{family:16s} K = 3: A and b match the published tableau exactly.")

Two structural identities hold for *any* collocation method, and they are cheap to check. They are
also the two mistakes that are easiest to make when typing a tableau by hand.

In [ ]:
for family in FAMILIES:
    for K in range(max(len(FAMILIES[family]["fixed"]), 1), 5):
        c, A, b, _ = butcher_tableau(family, K)

        # Row sums: sum_k a_{j,k} = c_j, because sum_k Omega_k(tau) = tau
        for j in range(K):
            assert sym.simplify(sum(A[j]) - c[j]) == 0, f"{family} K={K}: row {j + 1}"

        # Weights sum to one: the quadrature rule reproduces int_0^1 dtau = 1
        assert sym.simplify(sum(b) - 1) == 0, f"{family} K={K}: weights"

print("Row sums equal c and weights sum to 1 for every family and K tested.")

## Why $\bar\ell_j$ and not $\ell_j$

Here is the erratum, as a computation. Build $\Omega_j$ both ways for Gauss-Radau with $K = 3$ and
compare the degrees.

In [ ]:
K = 3
points = COLLOCATION_POINTS[("Gauss-Radau", K)]

# Correct: the K-point basis, tau_0 = 0 excluded
Omega_correct = omega_basis(points)

# Incorrect: the (K+1)-point basis, tau_0 = 0 included, as the PRINTED text has it
points_with_zero = [sym.Integer(0)] + list(points)
Omega_wrong = [
    sym.expand(sym.integrate(lagrange_basis(points_with_zero, j), (tau, 0, tau)))
    for j in range(1, K + 1)
]

print("degree of ellbar_j :", [sym.degree(lagrange_basis(points, j), tau) for j in range(K)])
print("degree of Omega_j (correct, ellbar under the integral):",
      [sym.degree(o, tau) for o in Omega_correct])
print("degree of Omega_j (wrong, ell under the integral)     :",
      [sym.degree(o, tau) for o in Omega_wrong])

# The book says Omega_j has degree K. Only the corrected definition delivers that.
assert all(sym.degree(o, tau) == K for o in Omega_correct)
assert all(sym.degree(o, tau) == K + 1 for o in Omega_wrong)

In [ ]:
b_wrong = [sym.simplify(o.subs(tau, 1)) for o in Omega_wrong]

print("b from ellbar (correct):", [float(x) for x in b])
print("b from ell    (wrong)  :", [float(x) for x in b_wrong])
print("sum of correct weights:", float(sum(b)))
print("sum of wrong weights:  ", float(sum(b_wrong)))

```{warning}
The wrong definition does not fail loudly. It produces a full tableau of plausible-looking numbers
that is simply not the Radau method, and a model built on it converges to the wrong answer. This is
why the errata are worth reading alongside the textbook rather than after a debugging session.
```

## Order of accuracy, measured

The truncation-error column of the family table is a claim about how many polynomial degrees the
quadrature rule integrates exactly. We can measure it: apply $\sum_k b_k f(\tau_k)$ to
$f(\tau) = \tau^d$ for increasing $d$ and find the first degree where it stops being exact.

In [ ]:
def highest_exact_degree(family, K, max_degree=12):
    """Highest polynomial degree the K-point quadrature rule integrates exactly on [0, 1]."""
    c, _, b, _ = butcher_tableau(family, K)
    for d in range(max_degree + 1):
        rule = sum(b[k] * c[k] ** d for k in range(K))
        exact = sym.Rational(1, d + 1)
        if sym.simplify(rule - exact) != 0:
            return d - 1
    return max_degree


print(f"{'Family':16s} {'K':>2s} {'exact through degree':>22s} {'expected':>10s}")
for family, expected in [
    ("Gauss-Legendre", lambda K: 2 * K - 1),
    ("Gauss-Radau", lambda K: 2 * K - 2),
    ("Gauss-Lobatto", lambda K: 2 * K - 3),
]:
    for K in range(max(len(FAMILIES[family]["fixed"]), 1), 5):
        d = highest_exact_degree(family, K)
        assert d == expected(K), f"{family} K={K}: got {d}, expected {expected(K)}"
        print(f"{family:16s} {K:2d} {d:22d} {expected(K):10d}")

The measured degrees are exactly $2K-1$, $2K-2$ and $2K-3$: each fixed endpoint costs one degree of
exactness. That is the price paid for Radau's stiff decay and for Lobatto's symmetric node set.

## Visualizing the basis polynomials

$\Omega_j(\tau)$ is the accumulated contribution of collocation point $j$ as we sweep across the
element. Reading the plot: every curve starts at zero (that is the $\int_0^\tau$), the value at
$\tau = \tau_j$ is the tableau entry $a_{j,k}$, and the value at $\tau = 1$ is the quadrature
weight $b_k$.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
tau_grid = np.linspace(0, 1, 201)

for ax, family in zip(axes, FAMILIES):
    c, _, b, Omega = butcher_tableau(family, 3)
    c_float = [float(p) for p in c]

    for k, poly in enumerate(Omega):
        f = sym.lambdify(tau, poly, "numpy")
        ax.plot(tau_grid, f(tau_grid), label=rf"$\Omega_{k + 1}$")
        ax.plot(1.0, float(b[k]), marker="o", color=ax.lines[-1].get_color())

    for point in c_float:
        ax.axvline(point, color="gray", linestyle=":", linewidth=1)

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"{family}, K = 3")
    ax.set_xlabel(r"$\tau$")

axes[0].set_ylabel(r"$\Omega_k(\tau)$")
axes[0].legend()
plt.tight_layout()
plt.show()

The dotted vertical lines are the collocation points and the markers at $\tau = 1$ are the
quadrature weights. Gauss-Lobatto's first point sits at $\tau_1 = 0$, which is why its first
tableau row is identically zero --- the method evaluates $f$ at the start of the element, where
nothing has accumulated yet.

## Takeaways

* The collocation points are **not tabulated constants**; they are the roots of a shifted
  Gauss-Jacobi polynomial, and the family is chosen by deciding which endpoints to fix.
* The Butcher coefficients are **not fitted**; they are $a_{j,k} = \Omega_k(\tau_j)$ and
  $b_k = \Omega_k(1)$, two evaluations of one integral.
* The basis under that integral is $\bar\ell_j$ (the $K$-point basis), not $\ell_j$. The degree
  check above is the fastest way to catch the error.
* Each fixed endpoint costs one degree of quadrature exactness, which is the arithmetic behind the
  $O(h^{2K})$, $O(h^{2K-1})$, $O(h^{2K-2})$ column.

In practice `Pyomo.DAE` builds these coefficients for you --- see [](./PyomoDAE_example.ipynb).
The point of deriving them once is to know what `scheme="LAGRANGE-RADAU"` actually selects.